# Demo: pixelated LR -> super-resolution output


In [5]:
from pathlib import Path
import sys

import torch
from omegaconf import OmegaConf
from PIL import Image
import matplotlib.pyplot as plt
from torchvision.transforms import functional as TF

# ensure project root on sys.path
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')


Project root: /home/and-petukhov/studies/efml-unet


In [8]:
from src.model import build_model

config = OmegaConf.load(PROJECT_ROOT / 'src' / 'configs' / 'baseline_x4.yaml')
ckpt_path = PROJECT_ROOT / 'checkpoints' / 'sr_unet_x4.pt'

device = 'cpu'
model = build_model(config).to(device)
state = torch.load(ckpt_path, map_location=device)
model.load_state_dict(state['model_state_dict'])
model.eval();
print(f'Loaded checkpoint: {ckpt_path}')


ConfigAttributeError: Missing key model
    full_key: model
    object_type=dict

In [ ]:
# Pick a high-resolution image
hr_path = PROJECT_ROOT / 'data' / 'sr' / 'val_hr' / '0844.png'
hr = Image.open(hr_path).convert('RGB')

# Create LR with the same degradation as training: bicubic down & up, scale x4
scale_down = 4
lr_small = hr.resize((hr.width // scale_down, hr.height // scale_down), Image.BICUBIC)
lr_upscaled = lr_small.resize(hr.size, Image.BICUBIC)

hr, lr_small, lr_upscaled


In [ ]:
# Run the SR model
lr_tensor = TF.to_tensor(lr_upscaled).unsqueeze(0).to(device)
with torch.no_grad():
    sr = model(lr_tensor).clamp(0.0, 1.0)

sr_img = TF.to_pil_image(sr.squeeze(0).cpu())


In [ ]:
# Visualize: LR small, pixelated input, SR prediction
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, title in zip(
    axes,
    [lr_small, lr_pixelated, sr_img],
    ['Low-res (nearest down)', 'Pixelated input (nearest up)', 'Model SR output'],
):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()
